# Tubes 2 IF3270 CNN, RNN & LSTM
**Dataset CNN**: Intel Image Classification (~25.000 gambar, 6 kelas)
**Dataset RNN/LSTM**: Flickr8k (image captioning)


In [ ]:
# =============================================================================
# GPU Setup ” jalankan PERTAMA sebelum import TF lain
# =============================================================================
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'[GPU] Ditemukan {len(gpus)} GPU: {[g.name for g in gpus]}')
    print(f'[GPU] TensorFlow versi: {tf.__version__}')
else:
    print('[GPU] Tidak ada GPU terdeteksi â€” menggunakan CPU.')
    print('      Untuk GPU NVIDIA di Windows, install: pip install tensorflow==2.10.0')
    print('      Atau gunakan WSL dan: pip install tensorflow[and-cuda]')
    print(f'[TF] Versi TensorFlow: {tf.__version__}')

print(f'[TF] Built with CUDA: {tf.test.is_built_with_cuda()}')

In [ ]:
# ── Colab: Mount Google Drive (skip jika lokal) ────────────────────────────────
import sys

IN_COLAB = 'google.colab' in sys.modules or 'google.colab' in str(type(None))
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('[Colab] Google Drive terpasang.')
else:
    print('[Lokal] Tidak di Colab — skip Drive mount.')

In [ ]:
# -- Colab: Clone repo dari GitHub -------------------------------------------
import sys, os

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

REPO_URL = 'https://github.com/danenftyessir/ChosaHeidan_Tubes-2_IF3270.git'
REPO_DIR = '/content/ChosaHeidan_Tubes-2_IF3270'

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        os.system(f'git clone {REPO_URL} {REPO_DIR}')
    else:
        os.system(f'git -C {REPO_DIR} pull')
    print(f'[Repo] Kode tersedia di {REPO_DIR}/src/')
else:
    print('[Lokal] Skip clone.')

In [ ]:
# ── Fix relative imports di shared/ (patch runtime, tidak perlu push repo) ────
import os, re

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    REPO_DIR = '/content/ChosaHeidan_Tubes-2_IF3270'
    _src = os.path.join(REPO_DIR, 'src')

    # 1. Buat __init__.py di semua package dir
    for _d in ['', 'shared', 'cnn', 'cnn/scratch', 'cnn/keras', 'cnn/utils', 'cnn/bonus',
               'lstm', 'lstm/scratch', 'lstm/keras', 'lstm/bonus',
               'rnn', 'rnn/scratch', 'rnn/keras', 'rnn/bonus']:
        _init = os.path.join(_src, _d, '__init__.py')
        if not os.path.exists(_init):
            open(_init, 'w').close()

    # ── Fungsi helper: reset patch lama, lalu terapkan yang benar ─────────────
    def _fix_file(path, bare_import, try_template_fn):
        with open(path, 'r') as f:
            c = f.read()

        # Hapus semua wrapping try/except yang mungkin broken/benar sebelumnya
        c = re.sub(
            r'(?m)^([ ]*)try:[ ]*\n[ ]*' + re.escape(bare_import.lstrip()) +
            r'[ ]*\n[ ]*except ImportError:[ ]*\n[^\n]*',
            lambda m: m.group(1) + bare_import.lstrip(),
            c
        )

        # Wrap ulang dengan indentasi yang benar (pakai regex per-baris)
        def _wrap(m):
            ind = m.group(1)
            return try_template_fn(ind)

        c = re.sub(
            r'^([ ]*)' + re.escape(bare_import.lstrip()) + r'$',
            _wrap,
            c,
            flags=re.MULTILINE
        )

        with open(path, 'w') as f:
            f.write(c)

    # 2. Fix dense.py: from .activations import get_activation
    _fix_file(
        os.path.join(_src, 'shared', 'dense.py'),
        'from .activations import get_activation',
        lambda ind: (f'{ind}try:\n'
                     f'{ind}    from .activations import get_activation\n'
                     f'{ind}except ImportError:\n'
                     f'{ind}    from activations import get_activation')
    )

    # 3. Fix intel_preprocess.py: from ..cnn.utils.utils import load_image
    _fix_file(
        os.path.join(_src, 'shared', 'intel_preprocess.py'),
        'from ..cnn.utils.utils import load_image',
        lambda ind: (f'{ind}try:\n'
                     f'{ind}    from ..cnn.utils.utils import load_image\n'
                     f'{ind}except ImportError:\n'
                     f'{ind}    from cnn.utils.utils import load_image')
    )

    print('[Fix] Patches applied — shared imports OK')
else:
    print('[Fix] Lokal — skip patch')

In [ ]:
import os, sys
import numpy as np

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

# ── Google Drive Folder IDs (fallback jika Drive API diperlukan) ───────────────
DRIVE_INTEL_ID  = '1Amb6Wi42esiEowKKAeEXalb22FHSEWne'
DRIVE_FLICKR_ID = '11TSyjWcx3mnp6lCXogtNnHrmSyxH5Mew'

if IN_COLAB:
    REPO_DIR = '/content/ChosaHeidan_Tubes-2_IF3270'
    SRC_DIR  = os.path.join(REPO_DIR, 'src')

    # Dataset langsung di root MyDrive (tanpa subfolder data/)
    _MYDRIVE = '/content/drive/MyDrive'
    DATA_DIR         = os.path.join(_MYDRIVE, 'intel_image_classification')
    FLICKR8K_DIR     = os.path.join(_MYDRIVE, 'flickr8k')
    WEIGHTS_DIR      = os.path.join(_MYDRIVE, 'weights', 'cnn')
    LSTM_WEIGHTS_DIR = os.path.join(_MYDRIVE, 'weights', 'lstm')
    RNN_WEIGHTS_DIR  = os.path.join(_MYDRIVE, 'weights', 'rnn')
    VOCAB_DIR        = os.path.join(_MYDRIVE, 'vocab')
    FEATURES_DIR     = os.path.join(_MYDRIVE, 'features')
    RESULTS_DIR      = os.path.join(_MYDRIVE, 'results')

    # ── Fallback: Drive API (authenticated, tanpa quota) ──────────────────────
    if not os.path.exists(DATA_DIR) or not os.path.exists(FLICKR8K_DIR):
        print('[Path] Folder tidak ditemukan di Drive mount — pakai Drive API...')
        from google.colab import auth
        auth.authenticate_user()
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaIoBaseDownload
        import io, concurrent.futures

        _svc = build('drive', 'v3')

        def _dl_folder(folder_id, dest, workers=6):
            os.makedirs(dest, exist_ok=True)
            items, page_token = [], None
            while True:
                resp = _svc.files().list(
                    q=f"'{folder_id}' in parents and trashed=false",
                    fields='nextPageToken,files(id,name,mimeType)',
                    pageToken=page_token, pageSize=1000
                ).execute()
                items.extend(resp.get('files', []))
                page_token = resp.get('nextPageToken')
                if not page_token:
                    break

            files = [(i, os.path.join(dest, i['name']))
                     for i in items
                     if i['mimeType'] != 'application/vnd.google-apps.folder']
            dirs  = [(i, os.path.join(dest, i['name']))
                     for i in items
                     if i['mimeType'] == 'application/vnd.google-apps.folder']

            def _one(pair):
                item, path = pair
                if os.path.exists(path):
                    return
                req = _svc.files().get_media(fileId=item['id'])
                with io.FileIO(path, 'wb') as fh:
                    dl = MediaIoBaseDownload(fh, req, chunksize=8 * 1024 * 1024)
                    done = False
                    while not done:
                        _, done = dl.next_chunk()

            with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as ex:
                list(ex.map(_one, files))
            for sub_item, sub_dest in dirs:
                _dl_folder(sub_item['id'], sub_dest, workers)

        if not os.path.exists(DATA_DIR):
            print('[Drive API] Mengunduh intel_image_classification ...')
            _dl_folder(DRIVE_INTEL_ID, DATA_DIR)

        if not os.path.exists(FLICKR8K_DIR):
            print('[Drive API] Mengunduh flickr8k ...')
            _dl_folder(DRIVE_FLICKR_ID, FLICKR8K_DIR)

else:
    SRC_DIR = os.path.abspath('.')
    if not os.path.exists(os.path.join(SRC_DIR, 'cnn')):
        SRC_DIR = os.path.join(os.path.abspath('.'), 'src')
    PROJECT_ROOT = os.path.dirname(SRC_DIR)

    DATA_DIR         = os.path.join(PROJECT_ROOT, 'data', 'intel_image_classification')
    FLICKR8K_DIR     = os.path.join(PROJECT_ROOT, 'data', 'flickr8k')
    WEIGHTS_DIR      = os.path.join(PROJECT_ROOT, 'weights', 'cnn')
    LSTM_WEIGHTS_DIR = os.path.join(PROJECT_ROOT, 'weights', 'lstm')
    RNN_WEIGHTS_DIR  = os.path.join(PROJECT_ROOT, 'weights', 'rnn')
    VOCAB_DIR        = os.path.join(PROJECT_ROOT, 'data', 'vocab')
    FEATURES_DIR     = os.path.join(PROJECT_ROOT, 'data', 'features')
    RESULTS_DIR      = os.path.join(PROJECT_ROOT, 'results')

sys.path.insert(0, SRC_DIR)
sys.path.insert(0, os.path.join(SRC_DIR, 'shared'))

for d in [WEIGHTS_DIR, LSTM_WEIGHTS_DIR, RNN_WEIGHTS_DIR,
          VOCAB_DIR, FEATURES_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'[Path] SRC_DIR         : {SRC_DIR}  (exists: {os.path.exists(SRC_DIR)})')
print(f'[Path] DATA_DIR        : {DATA_DIR}  (exists: {os.path.exists(DATA_DIR)})')
print(f'[Path] FLICKR8K_DIR    : {FLICKR8K_DIR}  (exists: {os.path.exists(FLICKR8K_DIR)})')
print(f'[Path] WEIGHTS_DIR     : {WEIGHTS_DIR}')
print(f'[Path] LSTM_WEIGHTS_DIR: {LSTM_WEIGHTS_DIR}')
print(f'[Path] RNN_WEIGHTS_DIR : {RNN_WEIGHTS_DIR}')

if IN_COLAB:
    assert os.path.exists(os.path.join(DATA_DIR, 'seg_train')), \
        f'seg_train/ tidak ada di: {DATA_DIR}'
    assert os.path.exists(os.path.join(DATA_DIR, 'seg_test')), \
        f'seg_test/ tidak ada di: {DATA_DIR}'
    assert os.path.exists(os.path.join(FLICKR8K_DIR, 'Images')), \
        f'flickr8k/Images/ tidak ada di: {FLICKR8K_DIR}'
    assert os.path.exists(os.path.join(FLICKR8K_DIR, 'captions.txt')), \
        f'flickr8k/captions.txt tidak ada di: {FLICKR8K_DIR}'
    print('[Path] Semua path OK')

## Bagian 1: Utility Functions (PIL/Pillow + NumPy)

In [ ]:
from shared.preprocessing import load_image, load_batch, extract_features

# Test load_image
# img = load_image('path/ke/gambar.jpg', target_size=(150, 150))
# print(f'Image shape: {img.shape}, dtype: {img.dtype}, range: [{img.min():.2f}, {img.max():.2f}]')

# Test load_batch
# batch = load_batch(['path1.jpg', 'path2.jpg'], target_size=(150, 150))
# print(f'Batch shape: {batch.shape}')  # (N, 150, 150, 3)

## Bagian 2: Forward Propagation From Scratch

In [ ]:
from cnn.scratch.conv2d import Conv2D
from cnn.scratch.locally_connected2d import LocallyConnected2D
from cnn.scratch.pooling import MaxPooling2D, AveragePooling2D, GlobalAveragePooling2D
from cnn.scratch.flatten import Flatten
from cnn.scratch.model_scratch import CNNScratch

## Bagian 3: Pelatihan Model (Keras)

Variasi hyperparameter (16 arsitektur):
- Jumlah layer konvolusi: [2, 4]
- Jumlah filter: [32, 128]
- Ukuran kernel: [(3,3), (5,5)]
- Pooling: ['max', 'average']

Total: 2 Ã— 2 Ã— 2 Ã— 2 = **16 arsitektur**

In [ ]:
from shared.intel_preprocess import IntelImagePreprocessor

preprocessor = IntelImagePreprocessor(DATA_DIR, target_size=(150, 150))
preprocessor.load_data()
preprocessor.summary()

In [ ]:
from cnn.keras.train import train_with_variations

results = train_with_variations(
    data_dir=DATA_DIR,
    arch_type='conv2d',
    layer_variations=[2, 4],
    filter_variations=[32, 128],
    kernel_variations=[(3, 3), (5, 5)],
    pooling_variations=['max', 'average'],
    epochs=30,
    batch_size=32,
    weights_dir=WEIGHTS_DIR,
    results_path=os.path.join(RESULTS_DIR, 'cnn_variations.json')
)

In [ ]:
# ── Ranking hasil variasi ──────────────────────────────────────────────────────
ranked = sorted(
    [(k, v) for k, v in results.items() if 'best_val_f1' in v],
    key=lambda x: x[1]['best_val_f1'],
    reverse=True
)

print(f'
{"="*60}')
print('  RANKING — Val Macro F1')
print(f'{"="*60}')
for i, (name, res) in enumerate(ranked, 1):
    print(f'  {i:2d}. {name:45s} F1={res["best_val_f1"]:.4f}')

if ranked:
    best_name = ranked[0][0]
    best_f1   = ranked[0][1]['best_val_f1']
    print(f'
  BEST : {best_name}  (F1={best_f1:.4f})')
    print(f'  Bobot: {WEIGHTS_DIR}/{best_name}.h5')

In [ ]:
# ── Verifikasi bobot tersimpan ─────────────────────────────────────────────────
saved = [f for f in os.listdir(WEIGHTS_DIR) if f.endswith('.h5')]
print(f'Bobot tersimpan ({len(saved)} file):')
for f in sorted(saved):
    size_mb = os.path.getsize(os.path.join(WEIGHTS_DIR, f)) / 1e6
    print(f'  {f}  ({size_mb:.1f} MB)')

## Bagian 4: Eksperimen dan Evaluasi

In [ ]:
from cnn.keras.evaluate import run_part4_evaluation

# Evaluasi dengan best model dari Bagian 3
# run_part4_evaluation(...)